# MNIST with Convolutional Networks

A LeNet-style CNN for handwritten digit classification. This notebook
focuses on the **type-safe spatial dimension chain** — the key feature
that makes idris-ml's CNN different from PyTorch.

```
Conv2d(1->16, k=5) -> ReLU -> MaxPool(2) ->
Conv2d(16->32, k=5) -> ReLU -> MaxPool(2) ->
Linear(512->10) (raw logits; loss applies log_softmax)
```

**CLI equivalent:** `make example-mnist` (requires `make download-mnist` first)


## Type-Safe Spatial Dimensions

In PyTorch, spatial dimension mismatches are runtime errors:
```python
# PyTorch: crashes at runtime if dimensions don't match
self.fc = nn.Linear(wrong_dim, 10)  # RuntimeError!
```

In idris-ml, `ConvOutDim` and `PoolOutDim` compute output dimensions
at the type level. A dimension mismatch is a **compile error**.


In [ ]:
:t conv2d

In [ ]:
:t maxPool2d

In [ ]:
:t ConvOutDim


In [ ]:
:t PoolOutDim


## Dimension Chain for MNIST

Starting from 28x28 images (1 channel):

| Layer | Output H | Output W | Channels | Flat dim |
|-------|----------|----------|----------|----------|
| Input | 28 | 28 | 1 | 784 |
| Conv2d(k=5) | 24 | 24 | 16 | 9216 |
| MaxPool(2) | 12 | 12 | 16 | 2304 |
| Conv2d(k=5) | 8 | 8 | 32 | 2048 |
| MaxPool(2) | 4 | 4 | 32 | 512 |
| Linear | - | - | - | 10 |

Each dimension is computed by the type system:
- `ConvOutDim 28 5 0 = 24` (28 - 5 + 1)
- `PoolOutDim 24 2 2 = 12` ((24 - 2) / 2 + 1)

If you change the kernel size or padding, the entire chain recomputes
at compile time, and any inconsistency is caught.


## Model Construction

Building the model uses the same `~>` chain as other architectures.
The type annotations on intermediate layers aren't required — they're
shown here for clarity.


In [ ]:
:exec run (do {
  m <- runInitL (conv2d {inC=1} {outC=16} {h=28} {w=28} {kH=5} {kW=5} {padH=0} {padW=0}
                  {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  discard m; liftIO1 (putStrLn "conv2d built (1x28x28 -> 16x24x24).") })

> **Note:** The cell above may produce a type constraint error in the REPL due to Idris 2's Peano Nat reduction limits at large dimensions. The CLI example (`make example-mnist`) compiles and runs correctly. See `docs/develop/gotchas.md` for details.


## Training on Real MNIST

This notebook demonstrates the architecture only. Real training
requires the MNIST dataset:

```bash
make download-mnist    # Downloads to data/mnist/
make example-mnist     # Trains LeNet, evaluates on test set
```

Expected results after ~100 epochs:
- Test accuracy: ~97%
- Uses `mkIndexedLoader` for shuffled mini-batch training
- Adam optimizer with gradient clipping

The data loading uses C FFI to read MNIST `.idx` binary files directly,
with `prim__mnistLoad`, `prim__mnistGetImage`, `prim__mnistGetLabel`.


## Also Available: 1D Convolutions

idris-ml also provides `conv1dLayer` and `maxPool1dLayer` for sequence
processing. The SeqClassify example uses them to classify waveforms
(see [seq_classify.ipynb](seq_classify.ipynb)).


In [ ]:
:t conv1d

## PyTorch Comparison

```python
class LeNet(nn.Module):
    def __init__(self):
        self.conv1 = nn.Conv2d(1, 16, 5)
        self.conv2 = nn.Conv2d(16, 32, 5)
        self.fc = nn.Linear(512, 10)  # 512 = 32 * 4 * 4

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, 512)  # <-- manual flatten, runtime error if wrong
        return self.fc(x)
```

The `x.view(-1, 512)` line is where PyTorch users commonly get runtime errors.
In idris-ml, `512 = OutC2 * (Pool2OutH * Pool2OutW)` is computed at the
type level and verified at compile time.

See `pytorch/torch_ref/scripts/mnist.py` for the full reference.


Next: [REINFORCE](reinforce.ipynb) — policy gradient reinforcement learning.
